# US_test Source-of-Truth Validation

This notebook uses only the network file in `saved/US_test/` as the single source of truth.
It checks metadata, bus country labels, and bus coordinates to determine whether the dataset is truly US.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa

network_path = Path('../saved/US_test/elec_s_min_ec_lcopt_Co2L-6h.nc')
if not network_path.exists():
    raise FileNotFoundError(f'Expected network not found: {network_path.resolve()}')

n = pypsa.Network(network_path)
print(f'Loaded: {network_path}')
print(f'Snapshots: {len(n.snapshots)} | Buses: {len(n.buses)} | Lines: {len(n.lines)} | Links: {len(n.links)}')

In [ ]:
meta = getattr(n, 'meta', {}) or {}
meta_countries = meta.get('countries', None)

bus_country_counts = pd.Series(dtype='int64')
if 'country' in n.buses.columns:
    bus_country_counts = (
        n.buses['country']
        .dropna()
        .astype(str)
        .str.upper()
        .value_counts()
    )

print('Meta countries:', meta_countries)
print('Bus country counts:')
print(bus_country_counts if not bus_country_counts.empty else 'No bus country labels')

In [ ]:
coords = n.buses[['x', 'y']].dropna().copy()
if coords.empty:
    raise ValueError('No bus coordinates found.')

lon = coords['x']
lat = coords['y']

bbox = {
    'lon_min': float(lon.min()),
    'lon_max': float(lon.max()),
    'lat_min': float(lat.min()),
    'lat_max': float(lat.max()),
}
centroid = {'lon': float(lon.mean()), 'lat': float(lat.mean())}

print('Coordinate bounding box:', bbox)
print('Coordinate centroid:', centroid)

# Very broad geographic envelopes for sanity checks.
# USA envelope includes mainland + Alaska + Hawaii + territories longitudes.
usa_env = {'lon_min': -180.0, 'lon_max': -60.0, 'lat_min': 15.0, 'lat_max': 75.0}
chn_env = {'lon_min': 73.0, 'lon_max': 136.0, 'lat_min': 18.0, 'lat_max': 54.0}

in_usa_envelope = (
    (lon >= usa_env['lon_min']) & (lon <= usa_env['lon_max']) &
    (lat >= usa_env['lat_min']) & (lat <= usa_env['lat_max'])
).mean()
in_chn_envelope = (
    (lon >= chn_env['lon_min']) & (lon <= chn_env['lon_max']) &
    (lat >= chn_env['lat_min']) & (lat <= chn_env['lat_max'])
).mean()

print(f'Fraction of buses inside USA envelope: {in_usa_envelope:.3f}')
print(f'Fraction of buses inside China envelope: {in_chn_envelope:.3f}')

In [ ]:
evidence = []

meta_us = isinstance(meta_countries, (list, tuple)) and ('US' in [str(c).upper() for c in meta_countries])
if meta_us:
    evidence.append('Metadata says US.')
else:
    evidence.append('Metadata does not say US.')

bus_majority = None
if not bus_country_counts.empty:
    bus_majority = str(bus_country_counts.index[0]).upper()
    evidence.append(f'Bus country majority is {bus_majority} ({int(bus_country_counts.iloc[0])} buses).')
else:
    evidence.append('No bus country labels available.')

if in_usa_envelope >= 0.95:
    evidence.append('Coordinates strongly match US envelope (>=95%).')
else:
    evidence.append('Coordinates do not match US envelope (less than 95%).')

if in_chn_envelope >= 0.95:
    evidence.append('Coordinates strongly match China envelope (>=95%).')
else:
    evidence.append('Coordinates do not strongly match China envelope.')

# Conservative verdict for high confidence.
is_us_confident = (in_usa_envelope >= 0.95) and (bus_majority in {None, 'US'})
is_not_us_confident = (in_usa_envelope < 0.05) and (in_chn_envelope >= 0.95) and (bus_majority == 'CN')

if is_us_confident:
    verdict = 'CONFIDENT VERDICT: This US_test network is US data.'
elif is_not_us_confident:
    verdict = 'CONFIDENT VERDICT: This US_test network is NOT US data (it is geographically consistent with China).'
else:
    verdict = 'MIXED VERDICT: Signals conflict; metadata and geography do not agree.'

print(verdict)
print('--- Evidence ---')
for item in evidence:
    print('-', item)